# M9 · Build BI marts

**Goal:** populate the three BI marts that feed the dashboard:
1. `marts.mart_fleet_daily` — day-grain executive KPI (sql/30)
2. `marts.mart_vehicle_monthly` — month-grain vehicle rollup (sql/31)
3. `marts.mart_tenant_monthly_summary` — month-grain tenant rollup of #1 + #2 (sql/32)

**Order matters.** Tenant summary depends on the other two, so it runs last.

**Recompute scope:** for a backfill we pass *all* dates / months currently represented in the warehouse. For incremental runs the orchestrator passes only `:touched_dates` / `:touched_months`.

In [1]:
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for c in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = c / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src)); break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, run_sql_file, transaction
from sqlalchemy import text
import pandas as pd

s = settings()
print(f'DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}')

DB = medali_dev@localhost:15432/accent_data


## 2. Inputs — derive the touched dates/months from the warehouse facts

In [2]:
# A backfill recomputes every date/month that has any fact rows.
with get_engine().connect() as conn:
    dates = pd.read_sql(text('''
        SELECT begin_path_time::DATE AS d FROM warehouse.fact_trip
        UNION SELECT notification_date FROM warehouse.fact_notification
        UNION SELECT maintenance_date FROM warehouse.fact_maintenance
        UNION SELECT fueling_date FROM warehouse.fact_fueling
        UNION SELECT event_time::DATE FROM warehouse.fact_harsh_event
        UNION SELECT stop_start::DATE FROM warehouse.fact_stop
        UNION SELECT begin_path_time::DATE FROM warehouse.fact_overspeed
    '''), conn)
TOUCHED_DATES = sorted(dates['d'].dropna().astype(str).tolist())
TOUCHED_MONTHS = sorted({d[:7] for d in TOUCHED_DATES})
print(f'dates: {len(TOUCHED_DATES):,}   months: {len(TOUCHED_MONTHS)}')
print('first/last month:', TOUCHED_MONTHS[:1], '...', TOUCHED_MONTHS[-1:])

dates: 2,384   months: 79
first/last month: ['2019-10'] ... ['2026-04']


## 3. Execute (in dependency order)

In [3]:
from accent_fleet.pipeline.run_log import begin_run, end_run
import time

run_id = begin_run(mode='notebook:05_build_bi_marts')
t0 = time.time(); written = {}
try:
    with transaction() as conn:
        r = run_sql_file(conn, '30_mart_fleet_daily.sql',
                         params={'touched_dates': TOUCHED_DATES, 'etl_run_id': run_id})
        written['fleet_daily'] = r.rowcount or 0
        r = run_sql_file(conn, '31_mart_vehicle_monthly.sql',
                         params={'touched_months': TOUCHED_MONTHS, 'etl_run_id': run_id})
        written['vehicle_monthly'] = r.rowcount or 0
        # tenant_summary depends on the two above — same transaction.
        r = run_sql_file(conn, '32_mart_tenant_monthly_summary.sql',
                         params={'touched_months': TOUCHED_MONTHS, 'etl_run_id': run_id})
        written['tenant_summary'] = r.rowcount or 0
    end_run(run_id, status='success', rows_loaded=sum(written.values()))
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc)); raise
print(f'done in {time.time()-t0:.1f}s', written)

done in 68.5s {'fleet_daily': 11883, 'vehicle_monthly': 24497, 'tenant_summary': 395}


## 4. Inspect

In [4]:
with get_engine().connect() as conn:
    head_fd = pd.read_sql(text('SELECT * FROM marts.mart_fleet_daily ORDER BY fleet_date DESC LIMIT 5'), conn)
    head_vm = pd.read_sql(text('SELECT * FROM marts.mart_vehicle_monthly ORDER BY year_month DESC, total_distance_km DESC LIMIT 5'), conn)
    head_ts = pd.read_sql(text('SELECT * FROM marts.mart_tenant_monthly_summary ORDER BY year_month DESC LIMIT 5'), conn)
display(head_fd); display(head_vm); display(head_ts)

,tenant_id,fleet_date,active_devices,total_trips,total_distance_km,total_driving_hours,avg_max_speed_kmh,overspeed_events,harsh_brake_events,harsh_accel_events,...,maintenance_alerts,other_alerts,total_stops,maintenance_events,maintenance_cost_total,fueling_events,fuel_litres,fuel_cost_total,_etl_run_id,_computed_at
0,235,2026-04-10,120,1290,12975.322142,381.884167,40.510078,162,503,1246,...,0,2,1430,0,0.0,0,0.0,0.0,36,2026-04-29 23:16:45.961188+00:00
1,238,2026-04-10,44,338,4388.604381,113.363611,43.517751,6,241,833,...,0,4,364,0,0.0,0,0.0,0.0,36,2026-04-29 23:16:45.961188+00:00
2,264,2026-04-10,47,408,8161.560164,177.028056,55.370098,636,42,26,...,0,384,442,0,0.0,0,0.0,0.0,36,2026-04-29 23:16:45.961188+00:00
3,1787,2026-04-10,1,1,0.070674,0.260833,5.000000,0,0,0,...,0,0,1,0,0.0,0,0.0,0.0,36,2026-04-29 23:16:45.961188+00:00
4,7486,2026-04-10,0,0,0.000000,NaN,NaN,0,1,1,...,0,0,5,0,0.0,0,0.0,0.0,36,2026-04-29 23:16:45.961188+00:00


,tenant_id,vehicle_id,year_month,active_days,total_trips,total_distance_km,total_driving_hours,trip_fuel_used_l,maintenance_events,maintenance_cost_total,...,reparation_events,reparation_amount_total,fueling_events,fuel_litres,fuel_cost_total,avg_cost_per_litre,cost_per_km,fuel_l_per_100km,_etl_run_id,_computed_at
0,264,99,2026-04,10,178,5275.141721,111.856944,0.0,0,0.0,...,0,0.0,0,0.0,0.0,None,0.0,None,36,2026-04-29 23:16:45.961188+00:00
1,264,34,2026-04,10,118,5079.493737,105.089167,0.0,0,0.0,...,0,0.0,0,0.0,0.0,None,0.0,None,36,2026-04-29 23:16:45.961188+00:00
2,264,91,2026-04,9,77,5076.053724,76.792500,0.0,0,0.0,...,0,0.0,0,0.0,0.0,None,0.0,None,36,2026-04-29 23:16:45.961188+00:00
3,264,95,2026-04,10,148,4693.989014,95.817778,0.0,0,0.0,...,0,0.0,0,0.0,0.0,None,0.0,None,36,2026-04-29 23:16:45.961188+00:00
4,264,49,2026-04,10,142,4491.100069,84.085556,0.0,0,0.0,...,0,0.0,0,0.0,0.0,None,0.0,None,36,2026-04-29 23:16:45.961188+00:00


,tenant_id,year_month,active_vehicles,active_devices,total_trips,total_distance_km,total_driving_hours,avg_distance_per_vehicle,total_overspeed,total_harsh_events,total_alerts,panic_alerts,total_maintenance_cost,total_fuel_cost,total_operating_cost,cost_per_km,_etl_run_id,_computed_at
0,7486,2026-04,0,0,0,0.000000,NaN,NaN,1598,33039,429,0,0.0,0.0,0.0,NaN,36,2026-04-29 23:16:45.961188+00:00
1,1787,2026-04,75,73,9638,82701.853285,2363.128611,1102.691377,50,32963,96,0,0.0,0.0,0.0,0.0,36,2026-04-29 23:16:45.961188+00:00
2,264,2026-04,57,54,5538,139161.548785,2966.442778,2441.430680,7051,4029,5831,0,0.0,0.0,0.0,0.0,36,2026-04-29 23:16:45.961188+00:00
3,238,2026-04,48,46,5203,75279.764881,1932.603056,1568.328435,49,21542,76,0,0.0,0.0,0.0,0.0,36,2026-04-29 23:16:45.961188+00:00
4,235,2026-04,134,126,15153,202029.395509,5326.830000,1507.682056,1422,26109,7,0,0.0,0.0,0.0,0.0,36,2026-04-29 23:16:45.961188+00:00
